# End-to-End Knowledge Graph Query Processing Pipeline

## Pipeline Overview
This notebook implements a comprehensive natural language to knowledge graph querying system that processes user questions and retrieves answers from an RDF knowledge graph stored in Turtle format.

---

## **STAGE 1: Query Understanding & Parsing**

### `query_understanding(user_query)`
**Purpose**: Converts natural language questions into structured subject-predicate-object format

**Input**:
- `user_query` (str): Natural language question from user

**Output**:
- Dictionary with relevant keys: `{'subject': '...', 'predicate': '...', 'object': '...'}`
- '?' to indicate the unknown component(s)

**LLM Usage**:
- **OpenAI GPT-4** with temperature=0.0 for deterministic parsing
- Uses structured prompting with examples to ensure consistent output format

**Key Features**:
- Automatically replaces spaces with underscores in entity names
- Identifies which component (subject/object/predicate) needs to be retrieved
- Handles various question types (Who, What, When, etc.)

---

## **STAGE 2: Entity Matching & Start Node Detection**

### `find_start_node(query_dict, model, kg_entities, entity_embeddings)`
**Purpose**: Identifies the known entity and finds its most semantically similar match in the knowledge graph

**Input**:
- `query_dict` (dict): Parsed query structure from Stage 1
- `model`: SentenceTransformer model (all-MiniLM-L6-v2)
- `kg_entities` (list): All entity labels extracted from the RDF graph
- `entity_embeddings` (tensor): Pre-computed embeddings for efficiency

**Output**:
- `start_node` (str): Best matching entity in the knowledge graph
- `modified_query` (dict): Updated query with the matched entity

**Key Features**:
- Uses cosine similarity for semantic matching
- Replaces only ONE known entity (subject or object)
- Handles entity name variations and formatting differences

---

## **STAGE 3: Subgraph Creation & Storage**

### `subgraph_creator(start_str, rdf_graph_path, max_depth=2)`
**Purpose**: Creates bidirectional subgraph around the start entity and saves it in multiple formats

**Input**:
- `start_str` (str): Starting entity name from Stage 2
- `rdf_graph_path` (str): Path to the main RDF graph file
- `max_depth` (int): Maximum traversal depth (default: 2)

**Output**:
- `nx_graph`: NetworkX DiGraph for visualization
- `triples` (list): List of tuples (subject, predicate, object)
- `subgraph`: RDFLib Graph object with extracted triples
- `turtle_filepath` (str): Path to saved Turtle file

**Key Features**:
- **Role Detection**: Identifies if entity exists as subject, predicate, or object
- **Bidirectional Traversal**: Explores both forward and reverse relationships
- **Multiple Starting Points**: Handles entities found in different roles simultaneously
- **Dual Storage**: Saves as both JSON and Turtle formats for different use cases

### Supporting Functions:

#### `save_triples_as_json(triples, start_str, rdf_graph_path)`
- **Input**: List of triples, entity name, graph path
- **Output**: JSON file path with triples in `{subject, predicate, object}` format

#### `save_subgraph_as_turtle(subgraph, start_str, rdf_graph_path)`
- **Input**: RDFLib Graph object, entity name, graph path
- **Output**: Turtle file path preserving RDF semantics

---

## **STAGE 4: SPARQL Query Generation with LLM Verification**

### `EmbedFirstLLMVerifier` Class
**Purpose**: Advanced SPARQL query generation using semantic similarity and LLM verification

#### Key Methods:

#### `_build_global_predicate_index()`
**Purpose**: Creates global vector index of all ontology predicates
- **Output**: FAISS index for efficient similarity search
- **Features**:
  - Extracts predicates with ns1: prefix
  - Combines IRI, labels, and descriptions for embedding
  - Uses cosine similarity with normalized embeddings

#### `_find_entity_subparts(entity)`
**Purpose**: Discovers related entities containing the start entity as substring
- **Input**: Entity name (str)
- **Output**: List of related entity URIs
- **Features**: Handles complex entity relationships and name variations

#### `_knn_search_predicates(query_phrase, k=20)`
**Purpose**: Performs semantic search when local predicates don't match
- **Input**: User's predicate phrase, number of results
- **Output**: Top-k predicates with similarity scores
- **Features**: Uses FAISS for efficient vector search

#### `_generate_llm_prompt(query_phrase, subject_type, top_predicates)`
**Purpose**: Creates structured prompt for LLM predicate selection
- **Input**: Query phrase, optional subject type, candidate predicates
- **Output**: Formatted prompt string
- **LLM Usage**: Designed for GPT-4 reasoning about semantic appropriateness

#### `_create_sparql_template(selected_predicates, known_entity, unknown_component)`
**Purpose**: Generates context-aware SPARQL queries
- **Input**: Selected predicates, known entity, unknown component type
- **Output**: Executable SPARQL query string
- **Features**:
  - Auto-detects entity forms (URI vs Literal)
  - Creates VALUES clauses for multiple predicates
  - Handles different unknown components (subject/object/predicate)

---

## **STAGE 5: Query Execution & Answer Generation**

### `run_sparql(g, sparql)`
**Purpose**: Executes SPARQL query against the RDF graph
- **Input**: RDFLib Graph object, SPARQL query string
- **Output**: List of binding dictionaries
- **Features**: Includes fallback mechanism for query execution failures

### `label_for(g, term)`
**Purpose**: Converts URIRefs to human-readable labels
- **Input**: RDFLib Graph, URI term or literal
- **Output**: Human-readable string representation
- **Features**:
  - Prioritizes RDFS labels
  - Handles URL encoding and underscores
  - Supports multiple languages (prefers English)

### `materialize_labels(g, rows)`
**Purpose**: Enriches SPARQL results with human-readable labels
- **Input**: RDFLib Graph, raw SPARQL results
- **Output**: List of dictionaries with both raw values and labels

### `llm_answer_from_bindings(question, labeled_rows, model="gpt-4")`
**Purpose**: Converts structured results into natural language answers
- **Input**: Original question, labeled results, model name
- **Output**: Natural language answer string
- **LLM Usage**:
  - **OpenAI GPT-4** with temperature=0 for consistent answers
  - Handles both successful results and no-result scenarios
  - Limits responses to available data only

---

## **STAGE 6: End-to-End Pipeline Integration Tester**

### `EndToEndPipelineTester` Class
**Purpose**: Basically, a tester function that orchestrates the complete pipeline and provides testing capabilities

#### Key Methods:

#### `run_complete_pipeline(natural_language_query, max_depth=2)`
**Purpose**: Executes all pipeline stages sequentially
- **Input**: Natural language question, optional depth parameter
- **Output**: Comprehensive results dictionary containing all stage outputs
- **Features**:
  - Error handling at each stage
  - Performance tracking
  - File path management

#### `print_pipeline_summary(results)`
**Purpose**: Provides detailed execution summary
- **Input**: Pipeline results dictionary
- **Output**: Formatted console output with statistics and generated queries

---

## **Data Flow Architecture**

```
Natural Language Query
         ↓
    Query Understanding
         ↓
    Entity Matching
         ↓
    Subgraph Creation  
         ↓
    SPARQL Generation  
         ↓
    Query Execution
         ↓
    Answer Generation
         ↓
    Natural Language Response
```

---

## **Technical Dependencies**

### **Models & APIs**:
- **OpenAI GPT-4**: Query parsing, predicate selection, answer generation
- **SentenceTransformer (all-MiniLM-L6-v2)**: Entity matching and semantic similarity
- **FAISS**: Efficient vector similarity search

### **Key Libraries**:
- **RDFLib**: RDF graph manipulation and SPARQL execution
- **NetworkX**: Graph visualization and analysis
- **PyTorch**: Tensor operations for embeddings
- **Pandas**: Results storage and analysis

### **File Formats**:
- **Input**: Turtle (.ttl) RDF files
- **Output**: JSON (triples), Turtle (subgraphs), CSV (results)

---

## **Configuration Parameters**

### **Embedding Model**:
- Model: `all-MiniLM-L6-v2`
- Dimension: 384
- Similarity: Cosine similarity with L2 normalization

### **RDF Namespaces**:
- Resources: `http://example.org/resource/`
- Predicates: `http://example.org/predicate/`

### **Search Parameters**:
- Max subgraph depth: 2-3 levels
- Top-k predicates: 20 (reduced to 10 for LLM)
- FAISS index type: IndexFlatIP (Inner Product)

---

## **Error Handling & Fallbacks**

### **Stage-Level Recovery**:
- Query parsing failures → Retry with simplified prompts
- Entity matching failures → Use original entity names
- SPARQL execution failures → Return informative error messages
- No results found → LLM generates helpful suggestions

### **Quality Assurance**:
- Validates SPARQL syntax before execution
- Checks entity existence in graph before subgraph creation
- Verifies file paths and permissions before saving
- Handles Unicode encoding issues in entity names

This pipeline provides a robust, scalable approach to natural language question answering over knowledge graphs with comprehensive error handling and multiple output formats for analysis and debugging.


In [1]:
# Mounting and installing the necessary pipeline packages during initial processing

# !pip install openai
!pip install rdflib
import openai
import os
import json
import re
import pandas as pd
import time
# mounting the google drive
from google.colab import drive
drive.mount('/content/drive')

# For Subgraph Creation
from rdflib import Graph, Namespace, URIRef
import networkx as nx

# Loading the Knowledge graph in turtle format
rdf_graph_path = "/path_to_file/output_graph.ttl"

# defining the openai API key
os.environ["OPENAI_API_KEY"] = 'sk-proj-**'
client = openai.OpenAI()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 10.2 MB/s eta 0:00:00
Mounted at /content/drive


## STAGE 1 : Query Deciphering

Implemented via an instruction-tuned LLM api token by OpenAI
#### Response Parameter Justification:
1. temperature=0.0 : ideal for Deterministic triple , and provide a stable EM/F1 score results

2. max_tokens=120: for short, bounded output,  prevents rambling and hallucination that breaks JSON parse.

3. system role: Stronger adherence to “exactly one ?”, underscores, simple predicates.

4. top_p=1.0 (set to default) : No nucleus sampling quirks; pairs well with temperature=0.


In [2]:
# Curating the query understanding function

def query_understanding(user_query):
  """
  input : (str) the user query in Natural Language
  return value : (dict) Dictionary of values which is the format of :
  { 'subject': "..",
    'predicate' : "..",
    'object' : "..",
  }

  """
  prompt = f"""Extract the subject, predicate, and object from the following question.

    CRITICAL RULES:
    1. Exactly ONE component must be '?' (the unknown we're looking for)
    2. Use underscores for multi-word terms (e.g., "Alex Ferguson" → "Alex_Ferguson")
    3. Keep predicates simple and standard (avoid overly specific details)

    DIRECTION PATTERNS:
    - "What [thing] [verb] [object]?" → subject='?', object=known_entity
    - "Who [verb] [object]?" → subject='?', object=known_entity
    - "What did [subject] [verb]?" → subject=known_entity, object='?'
    - "What is the relationship between X and Y?" → subject=X, predicate='?', object=Y

    EXAMPLES:

    Venue/Location Questions:
    Q: "What venue hosted the Ali vs. Liston fight?"
    A: {{'subject': '?', 'predicate': 'hosted_by', 'object': 'Ali_vs_Liston_fight'}}

    Q: "Where did the boxing match take place?"
    A: {{'subject': '?', 'predicate': 'location_of', 'object': 'boxing_match'}}

    Q: "Which location held the conference?"
    A: {{'subject': '?', 'predicate': 'held', 'object': 'conference'}}

    Person/Creator Questions:
    Q: "Who directed The Family Man?"
    A: {{'subject': '?', 'predicate': 'directed_by', 'object': 'The_Family_Man'}}

    Q: "Who composed the music for Inception?"
    A: {{'subject': '?', 'predicate': 'composed_music_for', 'object': 'Inception'}}

    Q: "Which person wrote Harry Potter?"
    A: {{'subject': '?', 'predicate': 'wrote', 'object': 'Harry_Potter'}}

    Relationship Questions:
    Q: "What is the relationship between Madonna and Minnie Mouse?"
    A: {{'subject': 'Madonna', 'predicate': '?', 'object': 'Minnie Mouse'}}

    Q: "How is Tesla connected to SpaceX?"
    A: {{'subject': 'Tesla', 'predicate': '?', 'object': 'SpaceX'}}

    Q: "What connects Oxford to education?"
    A: {{'subject': 'Oxford', 'predicate': '?', 'object': 'education'}}

    Action Questions:
    Q: "What did Enzo Maresca manage?"
    A: {{'subject': 'Enzo_Maresca', 'predicate': 'manage', 'object': '?'}}

    Q: "What teams did Jose Mourinho coach?"
    A: {{'subject': 'Jose_Mourinho', 'predicate': 'coach', 'object': '?'}}

    Q: "What did Ed Wood write?"
    A: {{'subject': 'Ed_Wood', 'predicate': 'wrote', 'object': '?'}}

    Sports Questions:
    Q: "Who scored against Sheffield Wednesday in 1977?"
    A: {{'subject': '?', 'predicate': 'scored_goal_against', 'object': 'Sheffield_Wednesday'}}

    Q: "Which team won the Premier League in 2020?"
    A: {{'subject': '?', 'predicate': 'won', 'object': 'Premier_League_2020'}}

    Temporal Questions:
    Q: "Who broke up with Dolores Fuller?"
    A: {{'subject': '?', 'predicate': 'broke_up_with', 'object': 'Dolores_Fuller'}}

    Q: "When was the iPhone first released?"
    A: {{'subject': 'iPhone', 'predicate': 'first_released', 'object': '?'}}

    Q: "What year did World War II end?"
    A: {{'subject': 'World_War_II', 'predicate': 'ended_in', 'object': '?'}}

    Business/Acquisition Questions:
    Q: "Which company acquired Instagram?"
    A: {{'subject': '?', 'predicate': 'acquired', 'object': 'Instagram'}}

    Q: "What did Google buy in 2006?"
    A: {{'subject': 'Google', 'predicate': 'bought', 'object': '?'}}

    Now extract from this question:
    Question: "{user_query}"
    Answer: """

  response = client.chat.completions.create(
      model="gpt-4",
      messages=[{"role": "user", "content": prompt}],
      temperature=0.0)
  # Extract and return dictionary
  answer = response.choices[0].message.content
  # print("Original Query:",eval(answer))
  # Use `ast.literal_eval` if security is a concern
  return eval(answer)



Visualisation of the Query Deciphering Function

In [3]:
# defined a set of 12 questions
questions_set_path = "/content/drive/MyDrive/JSON_PIPELINE/hotpot_small_dataset_extracted/hotpot_small_dataset/questions_set_easy.json"
with open(questions_set_path, "r", encoding="utf-8") as fh:
    question_items = json.load(fh)

    batch_results = []
    batch_qa_data = []

    for item in question_items:
        q = item["Question"]
        result = query_understanding(q)  # Your existing function call
        batch_qa_data.append({"Question": q, "Subject": result['subject'], "Predicate": result['predicate'], "Object": result['object']})

    # ADD THESE 2 LINES:
    df = pd.DataFrame(batch_qa_data)
    display(df)

,Question,Subject,Predicate,Object
0,Who directed The Family Man?,?,directed_by,The_Family_Man
1,Who scored a 97th-minute goal against Sheffiel...,?,scored_goal_against,Sheffield_Wednesday
2,Who won the President's Cup in 2007?,?,won,President's_Cup_2007
3,What venue hosted the Ali vs. Liston fight in ...,?,hosted,Ali_vs_Liston_fight_1965
4,The Colisée is what to Lewiston?,The_Colisée,?,Lewiston
5,What did Ed Wood do related to Plan 9 from Out...,Ed_Wood,do_related_to,?
6,What is the relationship of the Maine Nordique...,Maine_Nordiques,?,NAHL
7,Who broke up with Dolores Fuller?,?,broke_up_with,Dolores_Fuller
8,Which team did Matthew Wicks join?,Matthew_Wicks,join,?
9,What did Ed Wood write in 1965?,Ed_Wood,wrote,?


### Query Understanding Modification Required (For future Work Documentation) :-
#### Modification 1 : Adding a context key, so that the context is not lost while deciphering the query  

* Question: "Who scored a 97th-minute goal against Sheffield Wednesday?"
* Current output :
{'subject': '?', 'predicate': 'scored_goal_against', 'object': 'Sheffield_Wednesday'}
* enhanced output :
{
    'subject': '?',
    'predicate': 'scored_goal_against',
    'object': 'Sheffield_Wednesday',
    'context': {
        'time': '97th_minute',
        'event_type': 'goal',
        'constraints': ['time_specific', 'sports_event']
    }
}

#### Modification 2 : seamlessly render the format of questions where the user query is requesting for a relationship description between 2 or more entities. One of the greatest hiccups I faced while building this pipeline  
* Question: "What is the relationship of the Maine Nordiques to the NAHL?"
* {'subject': 'Maine_Nordiques', 'predicate': '?', 'object': 'NAHL'}

* {
    'subject': 'Maine_Nordiques',
    'NAHL',
    'predficate': 'relation',
    'object' : '?'
}

## STAGE 2 : Semantic Similarity Extractor for finding the `start_node`

- **Model Used** : `al-MiniLM-L6-v2`
- **Justification** : It is a 384-dim encoder, with a small parameter count (MiniLM, 6 layers). It runs fast on CPU and trivially on modest GPUs, which keeps this entity linking responsive and cheap—crucial because this step happens for every query. Currently, as the KG dimensions are not too much, this is an optimal model.

- I have refrained from setting a seed, as my results were always identical.
- Future implementations : determining the similarity extraction of multiple entities in the graph. This would be beneficial for multi-hop reasoning between multiple entities, when mentioned in the user query.

In [4]:
import torch
# Install compatible versions of transformers and sentence-transformers
# !pip install -U transformers==4.38.1 sentence-transformers==2.5.1
from sentence_transformers import SentenceTransformer

# Initializing the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# loading the saved universal knowledge graph
g = Graph()
g.parse(rdf_graph_path, format="turtle")
kg_entities = list(set(str(s).split('/')[-1]for s, p, o in g) |
                   set(str(o).split('/')[-1]for s, p, o in g) |
                   set(str(p).split('/')[-1]for s, p, o in g))

def find_start_node(query_dict, model = model, kg_entities = kg_entities, entity_embeddings= None):
    """
    Replace the most relevant node (subject if known, else object) with
    the most semantically similar KG entity label.

    Parameters
    ----------
    query_dict : dict
        {"subject": "...", "predicate": "...", "object": "..."}
    model : SentenceTransformer (or any .encode())
        Embedding model already loaded in memory.
    kg_entities : list[str]
        Labels/IRIs of entities present in the knowledge graph.
    entity_embeddings : torch.Tensor | None
        Pre-computed embeddings of `kg_entities` (shape N × d).  If None,
        they will be encoded on the fly.

    Returns
    -------
    dict : Same structure as `query_dict`, with at most ONE value changed.
    start_node : The node from where the subgraph begins
    """

    #  Pre-compute KG embeddings once if caller hasn’t supplied them
    if entity_embeddings is None:
        entity_embeddings = model.encode(kg_entities, convert_to_tensor=True)

    # Decide which node to replace
    node_key = "subject" if query_dict.get("subject") != "?" else "object"
    node_val = query_dict.get(node_key, "?")

    # No replacement if both are unknown
    if node_val == "?":
        return query_dict

    # Embed the node string
    node_embed = model.encode(node_val.replace("_", " "), convert_to_tensor=True)

    # Cosine similarity  method to determine the best KG label
    sims = torch.nn.functional.cosine_similarity(node_embed, entity_embeddings)
    # determine the best solution based on cosine similarity score
    best_idx = torch.argmax(sims).item()
    best_match = kg_entities[best_idx]
    start_node = best_match
    # Replace only that one field
    query_dict[node_key] = best_match

    # returns the subgraph starting node, and the query with the modified start node
    return start_node, query_dict


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## STAGE 3 : SubGraph Creator and Analyser

- **Implementation Library** : Python's networkx
- Documentation : https://networkx.org/documentation/stable/reference/index.html

This code has been created for the purpose of implementation over a turtle graph, for future modifications - it will be automated for more than 1 graph format (such as Neo4J, .gml format et cetera).

Prerequisite to this Code Cell :

The knowledge graph is stored in **Turtle (TTL) format**, which encodes data as RDF triples of the form* (subject, predicate, object)*. By W3C convention, subjects and predicates are URIs, while objects can be either URIs (other entities) or literals (strings, numbers, dates). The code binds two fixed namespaces (ex: for resources/entities and ns1: for predicates/relations) so that URIs in the graph map neatly to human-readable labels. Because RDF triples are inherently directional (subject → predicate → object), the traversal logic in the code distinguishes roles (subject, predicate, object) and uses this structure to explore local subgraphs bidirectionally.

---

Short Excerpt from the Graph for reference :
```
@prefix ex:  <http://example.org/resource/> .
@prefix ns1: <http://example.org/predicate/> .

ex:Hamlet      ns1:written_by      ex:William_Shakespeare .
ex:William_Shakespeare ns1:born_in "England" .
ex:Ophelia     ns1:loves           ex:Hamlet .
```

In [7]:
from collections import deque
import networkx as nx
from rdflib import Graph, URIRef, Literal, Namespace
import matplotlib.pyplot as plt
from collections import deque

# Initiliasing the prefixes, according to the current graph
# For the Subject prefix
EX = Namespace("http://example.org/resource/")
# For the Predicate prefix
ns1 = Namespace("http://example.org/predicate/")

def subgraph_creator(start_str, rdf_graph_path, max_depth=2):
    g = Graph()
    g.parse(rdf_graph_path, format="turtle")

    # initialising the subgraph
    subgraph = Graph()
    # Bind namespaces to the subgraph for cleaner output
    subgraph.bind("ex", EX)
    subgraph.bind("ns1", ns1)

    # ----- start-node variants : the node logic to be hard-coded for the purpose of understanding the nature of the node (to adhere to the URIRef formats for each node)
    subj_uri = URIRef(EX + start_str)              # subject style
    pred_uri = URIRef(ns1 + start_str)             # predicate style
    obj_uri = URIRef(EX + start_str)               # object as URI (same as subject URI)
    obj_lit = Literal(start_str)                   # literal object

    print(f"🔍 Searching for: {start_str}")
    # ----- role detection -----

    is_subj = any(g.predicate_objects(subject=subj_uri))
    is_pred = any(g.triples((None, pred_uri, None)))
    is_obj_uri = any(g.subject_predicates(object=obj_uri))
    is_obj_lit = any(g.subject_predicates(object=obj_lit))
    is_obj = is_obj_uri or is_obj_lit

    print(f"  Found as subject: {is_subj}")
    print(f"  Found as predicate: {is_pred}")
    print(f"  Found as object (URI): {is_obj_uri}")
    print(f"  Found as object (Literal): {is_obj_lit}")

    # ----- bfs queue: (node, depth, role) -----
    queue = deque()

    # Add all roles where the entity is found
    if is_subj:
        queue.append((subj_uri, 0, 'subject'))
        print(f"  Starting as SUBJECT: {subj_uri}")

    if is_pred:
        queue.append((pred_uri, 0, 'predicate'))
        print(f"  Starting as PREDICATE: {pred_uri}")

    if is_obj_uri:
        queue.append((obj_uri, 0, 'object'))
        print(f"  Starting as OBJECT (URI): {obj_uri}")

    if is_obj_lit:
        queue.append((obj_lit, 0, 'object'))
        print(f"  Starting as OBJECT (Literal): {obj_lit}")

    # If not found in any role, fallback to literal object
    if not (is_subj or is_pred or is_obj):
        queue.append((obj_lit, 0, 'object'))
        print(f"  Fallback as OBJECT (Literal): {obj_lit}")

    visited = set()
    triples = []
    nx_g = nx.DiGraph()
    nx_g.add_node(start_str)

    while queue:
        node, depth, role = queue.popleft()

        if depth > max_depth:
            continue

        # Use (node, role) as visited key to allow same node in different roles
        if (node, role) in visited:
            continue

        visited.add((node, role))

        # Get readable label
        if isinstance(node, URIRef):
            lab = str(node).split('/')[-1]
        else:
            lab = str(node)

        # ---------- role = predicate ----------
        if role == 'predicate':
            pred_count = 0
            for s, o in g.subject_objects(predicate=node):
                s_lab = str(s).split('/')[-1]
                o_lab = str(o).split('/')[-1] if isinstance(o, URIRef) else str(o)
                triples.append((s_lab, lab, o_lab))
                subgraph.add((s, node, o)) # Add triple to the subgraph rdflib graph
                nx_g.add_edge(s_lab, o_lab, label=lab, depth=depth, direction='pred')
                pred_count += 1

                if depth + 1 <= max_depth:
                    queue.append((s, depth + 1, 'subject'))
                    # For objects, determine if they should be treated as URIRef or literal
                    if isinstance(o, URIRef):
                        queue.append((o, depth + 1, 'subject'))  # URI objects can also be subjects
                    queue.append((o, depth + 1, 'object'))
            continue

        # ---------- always do reverse (find where current node is object) ----------
        reverse_count = 0
        for s, p in g.subject_predicates(object=node):
            s_lab = str(s).split('/')[-1]
            p_lab = str(p).split('/')[-1]
            triples.append((s_lab, p_lab, lab))
            # adding triples to the subgraph
            subgraph.add((s, p, node))
            nx_g.add_edge(s_lab, lab, label=p_lab, depth=depth, direction='reverse')
            reverse_count += 1

            if depth + 1 <= max_depth:
                queue.append((s, depth + 1, 'subject'))
                queue.append((p, depth + 1, 'predicate'))

        # ---------- forward traversal (find where current node is subject) ----------
        # This should work for both subjects and objects that are URIs
        forward_count = 0
        if isinstance(node, URIRef):  # Only URIRefs can be subjects
            for p, o in g.predicate_objects(subject=node):
                p_lab = str(p).split('/')[-1]
                o_lab = str(o).split('/')[-1] if isinstance(o, URIRef) else str(o)
                triples.append((lab, p_lab, o_lab))

                # Add triple to subgraph
                subgraph.add((node, p, o))

                nx_g.add_edge(lab, o_lab, label=p_lab, depth=depth, direction='forward')
                forward_count += 1

                if depth + 1 <= max_depth:
                    queue.append((p, depth + 1, 'predicate'))
                    # Add object to queue with appropriate role
                    if isinstance(o, URIRef):
                        queue.append((o, depth + 1, 'subject'))  # URI can be subject too
                    queue.append((o, depth + 1, 'object'))

    print(f"Final: {len(triples)} triples | {nx_g.number_of_nodes()} nodes | {nx_g.number_of_edges()} edges")

    # at this point, for every traversed graph, built upon the code for storage access in the triple
    # ----- Save triples to JSON file -----
    json_filepath = save_triples_as_json(triples, start_str, rdf_graph_path)
    turtle_filepath = save_subgraph_as_turtle(subgraph, start_str, rdf_graph_path)
    return nx_g, triples, subgraph, turtle_filepath # only turtle for now, will make it universal when the path works

def save_subgraph_as_turtle(subgraph, start_str, rdf_graph_path):
    """
    Save subgraph as Turtle format file named after the start_str

    Args:
        subgraph: rdflib.Graph object containing the subgraph
        start_str: Starting entity name (used for filename)
        rdf_graph_path: Path to RDF file (used to determine save location)

    Returns:
        str: Path to the saved Turtle file
    """
    # Create filename based on start_str
    safe_filename = start_str.replace(" ", "_").replace("/", "_").replace("\\", "_")
    turtle_filename = f"{safe_filename}_subgraph.ttl"

    # Determine save location (same directory as RDF file)
    rdf_dir = os.path.dirname(rdf_graph_path)
    turtle_filepath = os.path.join(rdf_dir, turtle_filename)

    try:
        # Save to Turtle file
        subgraph.serialize(destination=turtle_filepath, format='turtle')

        print(f"Subgraph saved to: {turtle_filepath}")
        print(f"Total triples in subgraph: {len(subgraph)}")

        return turtle_filepath

    except Exception as e:
        print(f"ERROR! Unable to save Turtle file: {e}")
        return None
def load_subgraph_from_turtle(turtle_filepath):
    """
    Load a subgraph from a Turtle file

    Args:
        turtle_filepath: Path to the Turtle file

    Returns:
        rdflib.Graph: Loaded subgraph
    """
    try:
        subgraph = Graph()
        # Bind namespaces
        subgraph.bind("ex", EX)
        subgraph.bind("ns1", ns1)

        subgraph.parse(turtle_filepath, format='turtle')
        print(f"Loaded subgraph with {len(subgraph)} triples from: {turtle_filepath}")
        return subgraph
    except Exception as e:
        print(f"ERROR! Unable to load the Turtle file: {e}")
        return None

def save_triples_as_json(triples, start_str, rdf_graph_path):
    """
    Save triples as JSON file named after the start_str

    Args:
        triples: List of tuples (subject, predicate, object)
        start_str: Starting entity name (used for filename)
        rdf_graph_path: Path to RDF file (used to determine save location)

    Returns:
        str: Path to the saved JSON file
    """
    # Convert triples to JSON format
    json_triples = []
    for subject, predicate, obj in triples:
        json_triples.append({
            "subject": subject,
            "predicate": predicate,
            "object": obj
        })

    # Create filename based on start_str
    safe_filename = start_str.replace(" ", "_").replace("/", "_").replace("\\", "_")
    json_filename = f"{safe_filename}.json"

    # Determine save location (same directory as RDF file)
    rdf_dir = os.path.dirname(rdf_graph_path)
    json_filepath = os.path.join(rdf_dir, json_filename)

    try:
        # Save to JSON file
        with open(json_filepath, 'w', encoding='utf-8') as f:
            json.dump(json_triples, f, indent=2, ensure_ascii=False)

        print(f"Triples saved to: {json_filepath}")
        print(f"Total triples saved: {len(json_triples)}")

        return json_filepath

    except Exception as e:
        print(f"ERROR! Unable to load the JSON file: {e}")
        return None
# This function was meant for visual reference for the developer, it was
def draw_nx_graph(nx_graph, start_node, query=None):
    """
    Visualize the bidirectional subgraph with proper styling.
    """
    if nx_graph.number_of_nodes() == 0:
        print("EMPTY GRAPH : Could not find any connections")
        return

    plt.figure(figsize=(16, 12))

    # Create layout with more space
    pos = nx.spring_layout(nx_graph, k=2, iterations=100, seed=42)

    # Color nodes by depth and highlight start node
    node_colors = []
    node_sizes = []

    for node in nx_graph.nodes():
        if node == start_node:
            node_colors.append('#e74c3c')  # Red for start node
            node_sizes.append(1500)
        else:
            # Get depth from edges
            depths = []
            for u, v, data in nx_graph.edges(data=True):
                if u == node or v == node:
                    depths.append(data.get('depth', 0))

            avg_depth = sum(depths) / len(depths) if depths else 0
            # depth distinction for the visualisation
            if avg_depth <= 1:
                node_colors.append('#3498db')  # Blue for depth 1
                node_sizes.append(1000)
            elif avg_depth <= 2:
                node_colors.append('#2ecc71')  # Green for depth 2
                node_sizes.append(800)
            else:
                node_colors.append('#f39c12')  # Orange for depth 3+
                node_sizes.append(600)

    # Draw nodes
    nx.draw_networkx_nodes(nx_graph, pos,
                          node_color=node_colors,
                          node_size=node_sizes,
                          alpha=0.8)

    # Draw edges with different colors for forward/reverse
    forward_edges = [(u, v) for u, v, d in nx_graph.edges(data=True) if d.get('direction') == 'forward']
    reverse_edges = [(u, v) for u, v, d in nx_graph.edges(data=True) if d.get('direction') == 'reverse']

    # Draw forward edges (blue)
    if forward_edges:
        nx.draw_networkx_edges(nx_graph, pos,
                              edgelist=forward_edges,
                              arrowstyle='->',
                              arrowsize=20,
                              edge_color='blue',
                              alpha=0.6,
                              width=2)

    # Draw reverse edges (green)
    if reverse_edges:
        nx.draw_networkx_edges(nx_graph, pos,
                              edgelist=reverse_edges,
                              arrowstyle='->',
                              arrowsize=20,
                              edge_color='green',
                              alpha=0.6,
                              width=2)

    # Draw node labels
    nx.draw_networkx_labels(nx_graph, pos, font_size=9, font_weight='bold')

    # Draw edge labels (predicates) - only show some to avoid clutter
    edge_labels = nx.get_edge_attributes(nx_graph, 'label')
    if len(edge_labels) <= 50:  # Only show labels if not too many
        nx.draw_networkx_edge_labels(nx_graph, pos,
                                    edge_labels=edge_labels,
                                    font_size=6,
                                    font_color='darkred')

    # Title and formatting
    title = f" BIDIRECTIONAL SUBGRAPH FOR'{start_node}'"
    if query:
        title += f"\nQuery: {query}"

    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')

    # Enhanced Legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=15, label='Start Node'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=12, label='Depth 1'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10, label='Depth 2'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=8, label='Depth 3+'),
        plt.Line2D([0], [0], color='blue', linewidth=3, label='Forward Edges'),
        plt.Line2D([0], [0], color='green', linewidth=3, label='Reverse Edges')
    ]
    plt.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()
    plt.show()

def analyze_query_subgraph(query, rdf_graph_path, max_depth=3):
    """
    Complete analysis pipeline for a query.
    """
    print("="*60)
    print("BIDIRECTIONAL SUBGRAPH ANALYSIS")
    print("="*60)

    start_node, unknown_position = find_start_node(query)

    print(f" Query: {query}")
    print(f" Initial node retrieved : {start_node}")
    print(f" Pipeline Looking for: {unknown_position}")
    print("-"*60)

    # Create subgraph
    nx_graph, triples = subgraph_creator(start_node, rdf_graph_path, max_depth)

    # # Show sample triples : in case if its called for
    # print(f"\nSample Triples from Subgraph:")
    # for i, (s, p, o) in enumerate(triples[:15]):  # Show first 15
    #     print(f"  {i+1:2d}. {s} --[{p}]--> {o}")

    # if len(triples) > 15:
    #     print(f"   {len(triples)-15} more triples")

    # Visualize
    draw_nx_graph(nx_graph, start_node, query)

    return nx_graph, triples


## STAGE 5 : SPARQL QUERY GENERATOR
---



## METHOD DESCRIPTIONS:

**1. Global Vector Index Building**
- Extracts all predicates with `ns1:` prefix
- Combines IRI, rdfs:label, skos:altLabel, and descriptions for embedding
- Uses FAISS for efficient similarity search

**2. Entity Subpart Discovery**
- `_find_entity_subparts()` finds entities containing your known entity as substring
- Automatically expands subgraph with these discovered entities
- Traverses 3 levels deep from each subpart entity

**3. k-NN Search (Step 2)**
- When user phrase isn't found locally, performs semantic search
- Returns top-k predicates with similarity scores
- Configurable k parameter (default 10)

**4. LLM Prompt Generation (Step 3)**
- Creates structured prompt with ranked predicates
- Includes subject type context when available
- Formats predicate info with labels, alt-labels, and descriptions

**5. SPARQL Template Creation (Step 4)**
- Generates `VALUES ?p { ... }` clause with selected predicates  
- Handles different unknown components (subject/object/predicate)
- Ready for back-off pattern implementation



About the function `EmbedFirstLLMVerifier`:
A class for verifying and constructing SPARQL queries using a hybrid embedding-based and LLM-driven approach. It loads an RDF graph in Turtle format, builds a semantic index of predicates, expands subgraphs based on entity variants, and generates SPARQL templates for answering natural language queries.

    Attributes
    ----------
    rdf_graph : rdflib.Graph
        The global RDF knowledge graph loaded from file.
    ns1_prefix : str
        The namespace prefix used for predicates (default: "http://example.org/predicate/").
    ns1 : rdflib.Namespace
        RDFLib Namespace object corresponding to `ns1_prefix`.
    resource_ns : rdflib.Namespace
        RDFLib Namespace for resource URIs.
    embedding_model : SentenceTransformer
        Pre-trained embedding model used for semantic similarity.
    predicate_metadata : List[dict]
        List of extracted predicate metadata including labels, alt labels, descriptions.
    faiss_index : faiss.Index
        Vector index storing predicate embeddings for k-NN retrieval.

In [11]:
import json
import numpy as np
import os
from typing import List, Dict, Tuple, Optional, Any, Union
from sentence_transformers import SentenceTransformer
!pip install faiss-cpu
import faiss
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, SKOS
import re

class EmbedFirstLLMVerifier:

    def __init__(self,
                 rdf_graph_path: str = "/content/drive/MyDrive/JSON_PIPELINE/output_graph.ttl",
                 model_name: str = "all-MiniLM-L6-v2",
                 ns1_prefix: str = "http://example.org/predicate/"):

        self.rdf_graph = Graph()
        self.rdf_graph.parse(rdf_graph_path, format='turtle')
        self.ns1_prefix = ns1_prefix
        self.ns1 = Namespace(ns1_prefix)
        self.resource_ns = Namespace("http://example.org/resource/")

        # Initialize embedding model
        self.embedding_model = SentenceTransformer(model_name)

        # Global predicate index
        self.predicate_index = None
        self.predicate_metadata = []
        self.faiss_index = None

        # Build global vector index on initialization
        self._build_global_predicate_index()

    def _build_global_predicate_index(self):
        """
        Build a FAISS vector index of all ontology predicates in the RDF graph.
        Returns: None
        """
        predicates_data = []

        # Extract all predicates with ns1: prefix
        for subj, pred, obj in self.rdf_graph:
            if str(pred).startswith(self.ns1_prefix):
                pred_info = self._extract_predicate_metadata(pred) # if any label metadata are provided ( which are not in this graph currently)
                if pred_info not in predicates_data:
                    predicates_data.append(pred_info)

        # Create embeddings
        texts_to_embed = []
        for pred_info in predicates_data:
            text_content = f"{pred_info['label']} {' '.join(pred_info['alt_labels'])} {pred_info['description']}"
            texts_to_embed.append(text_content.strip())

        # Generate embeddings : using the all-MiniLM-L6-v2 function
        embeddings = self.embedding_model.encode(texts_to_embed)
        # this set of code resembles the Foundational RAG code implementation to derive the vector embedding similarity to answer a question
        # Build FAISS index for the embeddings
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)

        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        self.faiss_index.add(embeddings.astype('float32'))

        self.predicate_metadata = predicates_data
        print(f"Built global predicate index with {len(predicates_data)} predicates")

    def _extract_predicate_metadata(self, predicate_uri: URIRef) -> Dict[str, Any]:
        """Extract comprehensive metadata for a predicate
        Note : not used in the graph currently, but added for making the code more inclusive of diverse turtle formats"""
        pred_info = {
            'iri': str(predicate_uri),
            'label': str(predicate_uri).split('/')[-1].replace('_', ' '),
            'alt_labels': [],
            'description': '',
            'domain': None,
            'range': None
        }

        # Extract RDFS label
        for obj in self.rdf_graph.objects(predicate_uri, RDFS.label):
            pred_info['label'] = str(obj)

        # Extract SKOS alternative labels
        for obj in self.rdf_graph.objects(predicate_uri, SKOS.altLabel):
            pred_info['alt_labels'].append(str(obj))

        # Extract description/comment
        for obj in self.rdf_graph.objects(predicate_uri, RDFS.comment):
            pred_info['description'] = str(obj)

        # Extract domain and range
        for obj in self.rdf_graph.objects(predicate_uri, RDFS.domain):
            pred_info['domain'] = str(obj)

        for obj in self.rdf_graph.objects(predicate_uri, RDFS.range):
            pred_info['range'] = str(obj)

        return pred_info

    def _find_entity_subparts(self, entity: str) -> List[str]:
        """Find entities that could be subparts of a longer entity, and also entities which have a part of the starting node,
         to increase the depth of the search """
        potential_subparts = []

        # Query for entities that contain the given entity as substring
        for subj, pred, obj in self.rdf_graph:
            if str(subj).startswith("http://example.org/resource/"):
                entity_name = str(subj).split('/')[-1].lower()
                if entity.lower() in entity_name and entity.lower() != entity_name:
                    potential_subparts.append(str(subj))

            if isinstance(obj, URIRef) and str(obj).startswith("http://example.org/resource/"):
                entity_name = str(obj).split('/')[-1].lower()
                if entity.lower() in entity_name and entity.lower() != entity_name:
                    potential_subparts.append(str(obj))
        return list(set(potential_subparts))

    def _expand_subgraph_with_subparts(self,
                                       subgraph: Graph,
                                       subpart_entities: List[str],
                                       depth: int = 1) -> Graph:
        """Expand subgraph with discovered subpart entities using Turtle format"""
        expanded_subgraph = Graph()

        for entity in subpart_entities:
            entity_uri = URIRef(entity)

            # Add triples where entity is subject (1 level deep only for the subgraph)
            self._traverse_from_entity(entity_uri, expanded_subgraph, depth=3, as_subject=True)

            # Add triples where entity is object (1 level deep only for the subgraph)
            self._traverse_from_entity(entity_uri, expanded_subgraph, depth=3, as_subject=False)

        # Add original subgraph triples to the expanded graph
        for triple in subgraph:
            expanded_subgraph.add(triple)

        original_count = len(subgraph)
        expanded_count = len(expanded_subgraph)
        added_count = expanded_count - original_count
        #print(f"Number of added triples to the Subgraph:{added_count}")
        return expanded_subgraph

    def _find_entity_variants_in_subgraph(self, entity_name: str, subgraph: Graph) -> List[Union[URIRef, Literal]]:
      """
      Find all variants of an entity within the subgraph
      Handles both URIRef subjects and Literal objects
      This function also performs the depth coverage increase essentially
      """
      print(f"Searching for variants of '{entity_name}' in subgraph")

      variants = []
      entity_lower = entity_name.lower()

      # Separate collection for better debugging
      uri_variants = []
      literal_variants = []

      # Get all unique entities from the subgraph
      all_subjects = set(subgraph.subjects())
      all_objects = set(subgraph.objects())

      #print(f"This Subgraph contains {len(all_subjects)} unique subjects, {len(all_objects)} unique objects")

      # Check subjects (usually URIRefs)
      for subj in all_subjects:
          if isinstance(subj, URIRef):
              # Extract local name from URI
              entity_str = str(subj)
              if '/' in entity_str:
                  local_name = entity_str.split('/')[-1]
              elif '#' in entity_str:
                  local_name = entity_str.split('#')[-1]
              else:
                  local_name = entity_str

              # Check if our entity name is contained in this local name
              if entity_lower in local_name.lower():
                  uri_variants.append(subj)
                  #for verification
                  #print(f"  Found URI subject variant: {local_name}")

          elif isinstance(subj, Literal):
              # Some subjects might be literals (rare but possible)
              literal_str = str(subj).lower()
              if entity_lower in literal_str:
                  literal_variants.append(subj)
                  #for verification
                  #print(f"  Found Literal subject variant: {subj}")

      # Check objects (can be URIRefs or Literals)
      for obj in all_objects:
          if isinstance(obj, URIRef):
              # Extract local name from URI
              entity_str = str(obj)
              if '/' in entity_str:
                  local_name = entity_str.split('/')[-1]
              elif '#' in entity_str:
                  local_name = entity_str.split('#')[-1]
              else:
                  local_name = entity_str

              # Check if our entity name is contained in this local name
              if entity_lower in local_name.lower():
                  uri_variants.append(obj)
                  # for verification
                  #print(f"  Found URI object variant: {local_name}")

          elif isinstance(obj, Literal):
              literal_str = str(obj).lower()
              if entity_lower in literal_str:
                  literal_variants.append(obj)
                  # for verification
                  # print(f"  Found Literal object variant: '{obj}'")

      # Combine all variants and remove duplicates
      variants = uri_variants + literal_variants
      unique_variants = []
      for variant in variants:
          if variant not in unique_variants:
              unique_variants.append(variant)

      print(f"Summary: {len(uri_variants)} URI variants, {len(literal_variants)} Literal variants")
      print(f"Total unique variants: {len(unique_variants)}")

      return unique_variants

    def _traverse_from_entity(self,
                              entity_uri: URIRef,
                              target_graph: Graph,
                              depth: int,
                              as_subject: bool):
        """Traverse graph from entity up to specified depth"""
        if depth <= 0:
            return

        if as_subject:
            # Entity as subject
            for pred, obj in self.rdf_graph.predicate_objects(entity_uri):
                target_graph.add((entity_uri, pred, obj))

                if isinstance(obj, URIRef) and depth > 1:
                    self._traverse_from_entity(obj, target_graph, depth-1, as_subject=True)

        else:
            for subj, pred in self.rdf_graph.subject_predicates(entity_uri):
                target_graph.add((subj, pred, entity_uri))
                if isinstance(subj, URIRef) and depth > 1:
                    self._traverse_from_entity(subj, target_graph, depth-1, as_subject=False)

    def _save_expanded_subgraph(self,
                                expanded_subgraph: Graph,
                                original_turtle_path: str) -> str:
        """Save the expanded subgraph as a new Turtle file"""

        # Create new filename for expanded subgraph
        base_name = os.path.splitext(os.path.basename(original_turtle_path))[0]
        expanded_filename = f"{base_name}_expanded.ttl"
        expanded_path = os.path.join(os.path.dirname(original_turtle_path), expanded_filename)

        try:
            # Save expanded subgraph
            expanded_subgraph.serialize(destination=expanded_path, format='turtle')
            #print(f"Expanded subgraph saved to: {expanded_path}")
            return expanded_path
        except Exception as e:
            print(f"ERROR : Could not save the expanded subgraph: {e}")
            return None

    def _extract_predicates_from_subgraph(self, subgraph: Graph) -> List[str]:
        """Extract all predicates from a subgraph"""
        predicates = set()
        for subj, pred, obj in subgraph:
            predicates.add(str(pred))
        return list(predicates)

    def _knn_search_predicates(self,
                               query_phrase: str,
                               k: int = 20) -> List[Dict[str, Any]]:
        """Once there are no local matches of the predicates found, Perform k-NN search on global predicate index"""

        # Embed the query phrase
        query_embedding = self.embedding_model.encode([query_phrase])
        faiss.normalize_L2(query_embedding)

        # Search - looking for the scores and indices
        scores, indices = self.faiss_index.search(query_embedding.astype('float32'), k)

        # Return top-k predicates with scores - returning top 10
        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(self.predicate_metadata):
                pred_info = self.predicate_metadata[idx].copy()
                pred_info['similarity_score'] = float(score)
                pred_info['rank'] = i + 1
                results.append(pred_info)

        return results

    def _generate_llm_prompt(self,
                             query_phrase: str,
                             subject_type: Optional[str],
                             top_predicates: List[Dict[str, Any]]) -> str:
        """Step 3: Generate prompt for LLM predicate selection"""

        predicate_descriptions = []
        for i, pred in enumerate(top_predicates, 1):
            desc = f"{i}. {pred['label']}"
            if pred['alt_labels']:
                desc += f" (also: {', '.join(pred['alt_labels'])})"
            if pred['description']:
                desc += f" - {pred['description']}"
            desc += f" [IRI: {pred['iri']}]"
            predicate_descriptions.append(desc)

        subject_context = f" for subject type '{subject_type}'" if subject_type else ""

        prompt = f"""You are helping to find the best ontology property that matches a user's natural language phrase.

          User phrase: "{query_phrase}"{subject_context}

          Available properties (ranked by semantic similarity):
          {chr(10).join(predicate_descriptions)}

          Choose the single property that BEST matches the user phrase. Consider:
          - Semantic meaning alignment
          - Subject type compatibility (if provided)
          - Property domain/range appropriateness

          Respond with only the property number (e.g., "3") and the corresponding IRI."""

        return prompt

    # ==================== CONTEXT-AWARE SPARQL METHODS ====================

    def _detect_entity_form_in_graph(self, entity_name: str) -> Optional[Union[URIRef, Literal]]:
        """
        Detect how an entity exists in the RDF graph
        Returns the actual form found in the graph (URIRef or Literal)
        """
        # Test different possible forms
        possible_forms = [
            # As subject URI
            URIRef(self.resource_ns + entity_name),
            # As predicate URI
            URIRef(self.ns1 + entity_name),
            # As object URI
            URIRef(self.resource_ns + entity_name),
            # As literal
            Literal(entity_name),
            # Try with URL encoding for spaces
            URIRef(self.resource_ns + entity_name.replace(" ", "_")),
            URIRef(self.ns1 + entity_name.replace(" ", "_")),
        ]

        for form in possible_forms:
            # Check if this form exists anywhere in the graph
            if isinstance(form, URIRef):
                # Check as subject
                if any(self.rdf_graph.predicate_objects(form)):
                    #print(f"Found '{entity_name}' as subject URI: {form}")
                    return form
                # Check as predicate
                if any(self.rdf_graph.subject_objects(form)):
                    #print(f"Found '{entity_name}' as predicate URI: {form}")
                    return form
                # Check as object
                if any(self.rdf_graph.subject_predicates(form)):
                    #print(f"Found '{entity_name}' as object URI: {form}")
                    return form
            else:  # Literal
                # Check as object literal
                if any(self.rdf_graph.subject_predicates(form)):
                    #print(f"Found '{entity_name}' as literal: {form}")
                    return form

        #print(f"Entity '{entity_name}' not found in graph in any standard form")
        return None

    def _format_entity_for_sparql(self, entity: Union[URIRef, Literal, str]) -> str:
        """Format an entity for SPARQL query based on its type"""
        if isinstance(entity, URIRef):
            return f"<{str(entity)}>"
        elif isinstance(entity, Literal):
            # Escape quotes in literals
            escaped_value = str(entity).replace('"', '\\"')
            return f'"{escaped_value}"'
        else:
            # If it's just a string, try to detect its form
            detected_form = self._detect_entity_form_in_graph(str(entity))
            if detected_form:
                return self._format_entity_for_sparql(detected_form)
            else:
                # Fallback: treat as literal
                escaped_value = str(entity).replace('"', '\\"')
                return f'"{escaped_value}"'

    def _detect_predicate_forms(self, predicate_list: List[str]) -> List[Union[URIRef, str]]:
        """Detect the correct form for each predicate in the list"""
        detected_predicates = []

        for pred_name in predicate_list:
            # Try to find the predicate in the graph
            detected_form = self._detect_entity_form_in_graph(pred_name.split('/')[-1] if '/' in pred_name else pred_name)

            if detected_form and isinstance(detected_form, URIRef):
                detected_predicates.append(detected_form)
            else:
                # If not found, use the full IRI as provided or construct it
                if pred_name.startswith('http://'):
                    pred_uri = URIRef(pred_name)
                else:
                    pred_uri = URIRef(self.ns1 + pred_name.replace(" ", "_"))
                detected_predicates.append(pred_uri)
                #print(f"Predicate '{pred_name}' using provided/constructed form: {pred_uri}")

        return detected_predicates
    def _create_sparql_template_with_variants(self,
                                            selected_predicates: List[str],
                                            entity_variants: List[Union[URIRef, Literal]],
                                            unknown_component: str,
                                            query_structure: Optional[Dict[str, str]] = None) -> str:
        """
        Create SPARQL template using multiple entity variants
        Replaces the original _create_sparql_template method
        """

        # print(f"Building SPARQL with {len(entity_variants)} entity variants")
        # print(f"  - Unknown component: '{unknown_component}'")

        if not entity_variants:
            print(" No entity variants provided. SPARQL Query may not be well-rendered")
            return "SELECT ?answer WHERE { # No variants found }"

        # Separate URIs and Literals for better handling
        uri_variants = [v for v in entity_variants if isinstance(v, URIRef)]
        literal_variants = [v for v in entity_variants if isinstance(v, Literal)]

        # print(f"  {len(uri_variants)} URI variants, {len(literal_variants)} Literal variants")

        # Format variants for SPARQL using existing method
        formatted_entity_variants = [self._format_entity_for_sparql(variant) for variant in entity_variants]

        # Handle predicates using existing method
        predicate_forms = self._detect_predicate_forms(selected_predicates)
        formatted_predicates = [self._format_entity_for_sparql(pred) for pred in predicate_forms]

        # Create VALUES clauses
        entity_values_clause = "VALUES ?entity { " + " ".join(formatted_entity_variants) + " }"
        predicate_values_clause = "VALUES ?p { " + " ".join(formatted_predicates) + " }"

        # print(f"Entity variants: {[str(v).split('/')[-1] if isinstance(v, URIRef) else str(v)[:20] for v in entity_variants]}")

        # Build query based on unknown component
        if unknown_component == "object":
            sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
    {entity_values_clause}
    {predicate_values_clause}
    ?entity ?p ?answer .
}}"""

        elif unknown_component == "subject":
            sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
    {entity_values_clause}
    {predicate_values_clause}
    ?answer ?p ?entity .
}}"""
        # for relationship entity questions : new addition
        else:  # predicate unknown
            if query_structure and query_structure.get('object') != '?':


                # Handle object entity variants using the working subgraph
                object_entity = query_structure['object']
                # We need access to working_subgraph here - see modification below

                # For now, use the existing detection method as fallback
                object_entity_form = self._detect_entity_form_in_graph(object_entity)
                if not object_entity_form:
                    object_entity_form = URIRef(self.resource_ns + object_entity.replace(" ", "_"))

                formatted_object_entity = self._format_entity_for_sparql(object_entity_form)

                sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
    {entity_values_clause}
    ?entity ?answer {formatted_object_entity} .
}}"""

            else:
                # General predicate query
                sparql_template = f"""SELECT DISTINCT ?p WHERE {{
    {entity_values_clause}
    {predicate_values_clause}
    ?entity ?p ?target .
}}"""

        print("Generated SPARQL with variants:")
        print(sparql_template)
        return sparql_template
    def _create_sparql_template(self,
                                    selected_predicates: List[str],
                                    known_entity: str,
                                    unknown_component: str,
                                    query_structure: Optional[Dict[str, str]] = None) -> str:


            """Generate a SPARQL query template that adapts to how entities actually exist in the graph"""

            print(f"🔧 Building context-aware SPARQL for:")
            print(f"  - Known entity: '{known_entity}'")
            print(f"  - Unknown component: '{unknown_component}'")
            print(f"  - Predicates: {selected_predicates[:3]}..." if len(selected_predicates) > 3 else f"  - Predicates: {selected_predicates}")
            if query_structure:
                print(f"  - Query structure: {query_structure}")
            print()

            # Extract entity name from full URI if needed
            if known_entity.startswith('http://'):
                entity_name = known_entity.split('/')[-1]
            else:
                entity_name = known_entity

            # Detect the correct form for the known entity
            known_entity_form = self._detect_entity_form_in_graph(entity_name)
            if not known_entity_form:
                # Fallback strategy: try common patterns
                if known_entity.startswith('http://'):
                    known_entity_form = URIRef(known_entity)
                else:
                    # Try as resource first, then as literal
                    resource_form = URIRef(self.resource_ns + entity_name.replace(" ", "_"))
                    if any(self.rdf_graph.predicate_objects(resource_form)) or any(self.rdf_graph.subject_predicates(resource_form)):
                        known_entity_form = resource_form
                        print(f"Using fallback resource form: {resource_form}")
                    else:
                        known_entity_form = Literal(entity_name)
                        print(f"Using fallback literal form: {known_entity_form}")

            # Detect correct forms for predicates
            predicate_forms = self._detect_predicate_forms(selected_predicates)

            # Format entities for SPARQL
            formatted_known_entity = self._format_entity_for_sparql(known_entity_form)
            formatted_predicates = [self._format_entity_for_sparql(pred) for pred in predicate_forms]

            # Create VALUES clause
            values_clause = "VALUES ?p { " + " ".join(formatted_predicates) + " }"

            # Build the query based on unknown component
            if unknown_component == "object":
                sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
        {values_clause}
        {formatted_known_entity} ?p ?answer .
    }}"""
            elif unknown_component == "subject":
                sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
        {values_clause}
        ?answer ?p {formatted_known_entity} .
    }}"""
            else:  # predicate unknown
                # ENHANCED: Handle relationship queries properly
                print(query_structure)
                if query_structure and query_structure.get('object') != '?':
                    #print("🔗 Detected relationship query - looking for specific predicate between two entities")

                    # Get the object entity from query structure
                    object_entity = query_structure['object']
                    print(f"  - Object entity: '{object_entity}'")

                    # Detect and format the object entity
                    object_entity_form = self._detect_entity_form_in_graph(object_entity)
                    if not object_entity_form:
                        # Fallback: construct URI form
                        object_entity_form = URIRef(self.resource_ns + object_entity.replace(" ", "_"))
                        print(f"  - Using fallback object URI: {object_entity_form}")
                    else:
                        print(f"  - Found object entity form: {type(object_entity_form).__name__}")

                    formatted_object_entity = self._format_entity_for_sparql(object_entity_form)

                    # Generate relationship-specific SPARQL (no VALUES clause needed)
                    sparql_template = f"""SELECT DISTINCT ?answer WHERE {{
        {formatted_known_entity} ?answer {formatted_object_entity} .
    }}"""
                    print(f" Relationship SPARQL: {formatted_known_entity} ?answer {formatted_object_entity}")

                else:
                    # Original logic for general predicate queries
                    #print("General predicate query - looking for any predicates from entity")
                    sparql_template = f"""SELECT DISTINCT ?p WHERE {{
        {values_clause}
        {formatted_known_entity} ?p ?target .
    }}"""

            print(" Generated SPARQL Query:")
            print(sparql_template)
            print("=" * 60)

            return sparql_template
    # ==================== MAIN PROCESS METHOD ====================
    # Updated process_query method
    def process_query(self,
                      user_query: str,
                      start_node: str,
                      subgraph_turtle_path: str,
                      unknown_component: str,
                      subject_type: Optional[str] = None,
                      query_structure: Optional[Dict[str, str]] = None) -> Dict[str, Any]:
        """Main function to process user query with entity variants support"""

        results = {
            'original_query': user_query,
            'start_node': start_node,
            'unknown_component': unknown_component,
            'subgraph_expanded': False,
            'sparql_queries': [],
            'llm_selections': [],
            'subpart_entities': [],
            "expanded_subgraph_path": None
        }

        # Load the original subgraph
        try:
            original_subgraph = Graph()
            original_subgraph.parse(subgraph_turtle_path, format='turtle')
            # print(f"Loaded original subgraph with {len(original_subgraph)} triples")
        except Exception as e:
            print(f"ERROR! Could not load the subgraph: {e}")
            return results

        # Check for entity subparts
        subpart_entities = self._find_entity_subparts(start_node)
        results['subpart_entities'] = subpart_entities

        # Expand subgraph if subparts found
        working_subgraph = original_subgraph
        if subpart_entities:
            working_subgraph = self._expand_subgraph_with_subparts(original_subgraph, subpart_entities, depth=1)
            results['subgraph_expanded'] = True

            # Save expanded subgraph
            expanded_path = self._save_expanded_subgraph(working_subgraph, subgraph_turtle_path)
            results['expanded_subgraph_path'] = expanded_path

            # print(f"Expanded subgraph with {len(subpart_entities)} subpart entities")

        # Find all entity variants in the (possibly expanded) subgraph
        entity_variants = self._find_entity_variants_in_subgraph(start_node, working_subgraph)

        if not entity_variants:
            # print(f"⚠️ No variants found for '{start_node}', using fallback")
            # Fallback to original entity detection
            fallback_form = self._detect_entity_form_in_graph(start_node)
            if fallback_form:
                entity_variants = [fallback_form]
            else:
                entity_variants = [URIRef(self.resource_ns + start_node.replace(" ", "_"))]

        # Extract predicates from the working subgraph
        local_predicates = self._extract_predicates_from_subgraph(working_subgraph)

        # Check if query phrase exists in local subgraph predicates
        query_words = set(user_query.lower().split())
        needs_knn = True
        for predicate in local_predicates:
            pred_words = set(predicate.lower().replace('_', ' ').split('/'))
            if query_words.intersection(pred_words):
                needs_knn = False
                break

        if needs_knn:
            # print(f"🔍 Query phrase '{user_query}' not found in local subgraph, performing k-NN search")

            # k-NN search
            top_predicates = self._knn_search_predicates(user_query, k=20)

            # Generate LLM prompt
            llm_prompt = self._generate_llm_prompt(user_query, subject_type, top_predicates[:10])
            results['llm_prompt'] = llm_prompt

            # Placeholder for LLM response
            selected_pred_iris = [pred['iri'] for pred in top_predicates[:3]]
            results['llm_selections'] = selected_pred_iris

            # NEW: Use entity variants in SPARQL template
            sparql_query = self._create_sparql_template_with_variants(
                selected_pred_iris,
                entity_variants,  # Pass all variants
                unknown_component,
                query_structure=query_structure
            )
            results['sparql_queries'].append({
                'query': sparql_query,
                'predicates_used': selected_pred_iris,
                'attempt': 1,
                'entity_variants_used': len(entity_variants)
            })

        else:
            # print("Query phrase found in local subgraph, using local predicates")

            #NEW: Use entity variants in SPARQL template
            sparql_query = self._create_sparql_template_with_variants(
                local_predicates,
                entity_variants,  # Pass all variants
                unknown_component,
                query_structure=query_structure
            )
            results['sparql_queries'].append({
                'query': sparql_query,
                'predicates_used': local_predicates,
                'attempt': 1,
                'source': 'local_subgraph',
                'entity_variants_used': len(entity_variants)
            })

        return results

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 49.8 MB/s eta 0:00:00


## STAGE 6 : GENERATING RESULTS FROM THE QUERY

In [18]:
# --- 1) Enhanced SPARQL : bindings with Fallback Strategies ---------------------------------
from rdflib import Graph, URIRef, RDFS, Literal
import re

def run_sparql(g: Graph, sparql: str, known_entity: str = None, unknown_component: str = None):
    """
    Enhanced SPARQL execution with 3-strategy fallback:
    1. Original SPARQL
    2. Bidirectional search
    3. Object literal reversal
    """

    # Strategy 1: Try Original SPARQL
    #print("Strategy 1: Original SPARQL")
    original_results = _try_original_sparql(g, sparql)
    if original_results:
        print(f"SPARQL Query generated {len(original_results)} results")
        return original_results

    print(" ERROR !Original SPARQL returned 0 results")

    # Only try fallbacks if we have entity information
    if not known_entity or not unknown_component:
        print("⚠️ No entity info for fallbacks, stopping here")
        return []

    # Strategy 2: Bidirectional Search
    #print("Strategy 2: Bidirectional search")
    bidirectional_results = _try_bidirectional_search(g, known_entity, unknown_component)
    if bidirectional_results:
        #print(f"Bidirectional search succeeded: {len(bidirectional_results)} results")
        return bidirectional_results

    #print("Bidirectional search returned 0 results")

    # Strategy 3: Object Literal Reversal
    #print("Strategy 3: Object literal reversal")
    reversal_results = _try_object_literal_reversal(g, known_entity, unknown_component)
    if reversal_results:
        #print(f"✅ Object literal reversal succeeded: {len(reversal_results)} results")
        return reversal_results

    #print("Object literal reversal returned 0 results")
    print("All strategies failed, could not retrieve any results")
    return []

def _try_original_sparql(g: Graph, sparql: str):
    """Strategy 1: Execute the original SPARQL as-is"""
    try:
        res = g.query(sparql)
        rows = []
        for r in res:
            val = r.get("answer")  # Assumes ?answer variable
            if val is not None:
                rows.append({"answer": val})
        return rows
    except Exception as e:
        print(f"   Original SPARQL execution failed: {e}")
        return []

def _try_bidirectional_search(g: Graph, known_entity: str, unknown_component: str):
    """
    Strategy 2: Try searching in both directions with both URI and literal forms
    """

    # Prepare both URI and literal forms of the entity
    entity_uri = f"<http://example.org/resource/{known_entity.replace(' ', '_')}>"
    entity_literal = f'"{known_entity}"'

    bidirectional_queries = []

    if unknown_component == "object":
        # Looking for objects - try entity as subject in both forms
        bidirectional_queries = [
            f"SELECT DISTINCT ?answer WHERE {{ {entity_uri} ?p ?answer . }}",
            f"SELECT DISTINCT ?answer WHERE {{ {entity_literal} ?p ?answer . }}",
            # Also try reverse (entity as object)
            f"SELECT DISTINCT ?answer WHERE {{ ?answer ?p {entity_uri} . }}",
            f"SELECT DISTINCT ?answer WHERE {{ ?answer ?p {entity_literal} . }}"
        ]

    elif unknown_component == "subject":
        # Looking for subjects - try entity as object in both forms
        bidirectional_queries = [
            f"SELECT DISTINCT ?answer WHERE {{ ?answer ?p {entity_uri} . }}",
            f"SELECT DISTINCT ?answer WHERE {{ ?answer ?p {entity_literal} . }}",
            # Also try reverse (entity as subject)
            f"SELECT DISTINCT ?answer WHERE {{ {entity_uri} ?p ?answer . }}",
            f"SELECT DISTINCT ?answer WHERE {{ {entity_literal} ?p ?answer . }}"
        ]

    elif unknown_component == "predicate":
        # Looking for predicates - try entity in both subject and object positions
        bidirectional_queries = [
            f"SELECT DISTINCT ?answer WHERE {{ {entity_uri} ?answer ?o . }}",
            f"SELECT DISTINCT ?answer WHERE {{ ?s ?answer {entity_uri} . }}",
            f"SELECT DISTINCT ?answer WHERE {{ {entity_literal} ?answer ?o . }}",
            f"SELECT DISTINCT ?answer WHERE {{ ?s ?answer {entity_literal} . }}"
        ]

    # Try each bidirectional query
    for i, query in enumerate(bidirectional_queries, 1):
        print(f"   Trying bidirectional query {i}: {_truncate_query(query)}")
        try:
            res = g.query(query)
            rows = []
            for r in res:
                val = r.get("answer")
                if val is not None:
                    rows.append({"answer": val})

            if rows:
                print(f"  Bidirectional query {i} succeeded with {len(rows)} results")
                return rows
            else:
                print(f"Bidirectional query {i} returned 0 results")

        except Exception as e:
            print(f"Bidirectional query {i} failed: {e}")

    return []

def _try_object_literal_reversal(g: Graph, known_entity: str, unknown_component: str):
    """
    Strategy 3: Specifically handle object literal reversal cases
    This addresses the core issue where entity exists as object literal but query treats it as subject
    """

    print(f"   Checking if '{known_entity}' exists as object literal...")

    # First, verify that the entity actually exists as an object literal
    literal_check_query = f'SELECT ?s ?p WHERE {{ ?s ?p "{known_entity}" . }}'

    try:
        literal_check_res = g.query(literal_check_query)
        literal_exists = len(list(literal_check_res)) > 0

        if not literal_exists:
            #print(f"   Entity '{known_entity}' not found as object literal, skipping reversal")
            return []
        else:
          print()
            #print(f"   ✅ Entity exists as object literal, proceeding with reversal")

    except Exception as e:
        print()
        #print(f"   Error checking literal existence: {e}")
        return []
    return []

    # Generate reversal queries based on unknown component
    reversal_queries = []

    if unknown_component == "object":
        # Original would be: "entity" ?p ?answer
        # Reversal: ?answer ?p "entity" (find what points TO the literal)
        reversal_queries = [
            f'SELECT DISTINCT ?answer WHERE {{ ?answer ?p "{known_entity}" . }}',
            # Also try finding related entities through intermediate connections
            f'SELECT DISTINCT ?answer WHERE {{ ?intermediate ?p "{known_entity}" . ?intermediate ?related_p ?answer . FILTER(?related_p != ?p) }}'
        ]

    elif unknown_component == "subject":
        # Original would be: ?answer ?p "entity"
        # This is already correct for literals, but let's try variations
        reversal_queries = [
            f'SELECT DISTINCT ?answer WHERE {{ ?answer ?p "{known_entity}" . }}',
            # Try finding subjects of subjects that point to this literal
            f'SELECT DISTINCT ?answer WHERE {{ ?answer ?intermediate_p ?intermediate . ?intermediate ?p "{known_entity}" . }}'
        ]

    elif unknown_component == "predicate":
        # Find predicates that connect to this literal
        reversal_queries = [
            f'SELECT DISTINCT ?answer WHERE {{ ?s ?answer "{known_entity}" . }}',
            f'SELECT DISTINCT ?answer WHERE {{ "{known_entity}" ?answer ?o . }}'
        ]

    # Try each reversal query
    for i, query in enumerate(reversal_queries, 1):
        print(f"   Trying reversal query {i}: {_truncate_query(query)}")
        try:
            res = g.query(query)
            rows = []
            for r in res:
                val = r.get("answer")
                if val is not None:
                    rows.append({"answer": val})

            if rows:
                #print(f"   ✅ Reversal query {i} succeeded with {len(rows)} results")
                return rows
            else:
              print()
        except:
          print()
    return []

def _truncate_query(query: str, max_length: int = 80) -> str:
    """Helper function to truncate long queries for logging"""
    if len(query) <= max_length:
        return query
    return query[:max_length] + "..."

# --- 2) Get readable labels (unchanged) --------------------------------
def label_for(g: Graph, term):
    """Get the answer retrieved from URIRefs - only used for RDFlib"""
    if isinstance(term, URIRef):
        # try rdfs:label
        for lbl in g.objects(term, RDFS.label):
            if isinstance(lbl, Literal):
                # prefer english if present
                if lbl.language in (None, "en"):
                    return str(lbl)
        # Enhanced fallback: handle URL encoding
        uri_str = str(term)
        fragment = uri_str.rsplit("/", 1)[-1].replace("_", " ")
        # Handle URL encoding like %27 for apostrophes
        import urllib.parse
        return urllib.parse.unquote(fragment)
    return str(term)

def materialize_labels(g: Graph, rows):
    """
    Taking the SPARQL query results (rows) and turning them into a list of dictionaries where each dictionary contains:
    "answer_value" :- the raw value of the SPARQL result (as a string).
    "answer_label" :-  a human-readable label for that value, looked up from the RDF graph g.
    """
    out = []
    for r in rows:
        v = r["answer"]
        out.append({
            "answer_value": str(v),
            "answer_label": label_for(g, v)
        })
    return out

def llm_answer_from_bindings(question: str, labeled_rows: list, model="gpt-4"):
    print(f'labelled rows{labeled_rows}')
    if not labeled_rows:
      # Let the LLM explain no-result or suggest next steps
        prompt = (f"Question: {question}\n"
                  "No results were returned by the SPARQL query."
                  "In one short sentence, say that no facts were found and suggest trying a different phrasing or direction.")
        r = client.chat.completions.create(
            model=model, temperature=0,
            messages=[{"role":"user","content":prompt}]
        )
        return r.choices[0].message.content.strip()


    candidates = "\n".join(f"- {r['answer_label']}" for r in labeled_rows[:10])
    # final prompt drafted
    prompt = (f"Question: {question}\n\n"
              f"Available answers from the knowledge base:\n{candidates}\n\n"
              f"Based on the information provided, answer the question directly and clearly. "
              f"Use the exact information given. If there's only one answer, state it clearly. "
              f"If there are multiple answers, list them appropriately. "
              f"Answer in a natural, complete sentence.")

    r = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role":"user","content":prompt}]
    )
    return r.choices[0].message.content.strip()
# --- 4) Enhanced Glue Function -----------------------------------
def answer_question_with_sparql(g: Graph, question: str, sparql: str,
                               known_entity: str = None, unknown_component: str = None):
    """
    Enhanced version that passes entity information to run_sparql for fallback strategies
    """
    rows = run_sparql(g, sparql, known_entity, unknown_component)
    labeled = materialize_labels(g, rows)
    final = llm_answer_from_bindings(question, labeled)
    return {
        "question": question,
        "sparql": sparql,
        "rows": labeled,
        "answer": final,
        "strategy_used": "enhanced_traversal_fallback"
    }


## PIPELINE-EXECUTION

In [20]:
import json
import os
import pandas as pd  # Added pandas import
from typing import Dict, Any, Optional, List
from rdflib import Graph, URIRef, Namespace
from rdflib.namespace import RDF, RDFS, SKOS
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class EndToEndPipelineTester:
    def __init__(self, rdf_graph_path: str):
        """
        Initialize the end-to-end pipeline tester

        Args:
            rdf_graph_path: Path to the RDF graph file
        """
        self.rdf_graph_path = rdf_graph_path

        # Load the RDF graph
        self.rdf_graph = Graph()
        self.rdf_graph.parse(rdf_graph_path, format="turtle")

        # Initialize the fixed EmbedFirstLLMVerifier with the loaded graph
        self.llm_verifier = EmbedFirstLLMVerifier(
            rdf_graph_path=self.rdf_graph_path,
            model_name="all-MiniLM-L6-v2",
            ns1_prefix="http://example.org/predicate/"
        )

        print(f"Pipeline initialized with graph containing {len(self.rdf_graph)} triples")

    def load_subgraph_turtle(self, turtle_filepath: str) -> Graph:
        """
        Load subgraph from Turtle file

        Args:
            turtle_filepath: Path to the Turtle file containing subgraph

        Returns:
            rdflib.Graph object containing the subgraph
        """
        try:
            subgraph = Graph()
            subgraph.parse(turtle_filepath, format='turtle')
            #print(f"Loaded subgraph with {len(subgraph)} triples from {turtle_filepath}")
            return subgraph
        except Exception as e:
           # print(f"Error! loading subgraph Turtle: {e}")
            return Graph()

    def run_complete_pipeline(self, natural_language_query: str, max_depth: int = 2) -> Dict[str, Any]:
        """
        Run the complete end-to-end pipeline

        Args:
            natural_language_query: Natural language query from user
            max_depth: Maximum depth for subgraph creation

        Returns:
            Complete results from all pipeline stages
        """
        print("="*80)
        print("STARTING END-TO-END PIPELINE TESTING")
        print("="*80)
        print(f"Natural Language Query: {natural_language_query}")
        print()

        pipeline_results = {
            'input_query': natural_language_query,
            'query_understanding': {},
            'entity_matching': {},
            'subgraph_creation': {},
            'sparql_generation': {},
            'turtle_files': {},
            'final_answer': '',  # Added for DataFrame
            'error': None        # Added for DataFrame
        }

        try:
            # STAGE 1: Query Understanding
            print("STAGE 1: Query Understanding")
            print("-" * 40)

            query_dict = query_understanding(natural_language_query)
            pipeline_results['query_understanding'] = query_dict

            print(f"Parsed query: {query_dict}")
            print()

            # STAGE 2: Entity Matching & Start Node Detection
            print("STAGE 2: Entity Matching & Start Node Detection")
            print("-" * 40)

            start_node, modified_query = find_start_node(query_dict)
            pipeline_results['entity_matching'] = {
                'start_node': start_node,
                'modified_query': modified_query
            }

            #print(f"Start node: {start_node}")
            print(f"Modified query: {modified_query}")
            print()

            # STAGE 3: Subgraph Creation
            print("STAGE 3: Subgraph Creation")
            print("-" * 40)

            nx_graph, triples, subgraph, turtle_filepath = subgraph_creator(start_node, self.rdf_graph_path, max_depth)

            safe_filename = start_node.replace(" ", "_").replace("/", "_").replace("\\", "_")
            json_filename = f"{safe_filename}.json"
            rdf_dir = os.path.dirname(self.rdf_graph_path)
            json_filepath = os.path.join(rdf_dir, json_filename)

            pipeline_results['subgraph_creation'] = {
                'nx_graph': nx_graph,
                'triples': triples,
                'turtle_file': turtle_filepath,
                'num_triples': len(triples)
            }

            # print(f"Created subgraph with {len(triples)} triples")
            # print(f"Saved to: {turtle_filepath}")
            print()

            # STAGE 4: Load Subgraph and Run SPARQL Generation
            print(" STAGE 4: SPARQL Query Generation with LLM Verification ")
            print("-" * 40)

            # Determine unknown component and subject type
            unknown_component = None
            subject_type = None

            if query_dict['subject'] == '?':
                unknown_component = 'subject'
                if 'person' in natural_language_query.lower() or 'who' in natural_language_query.lower():
                    subject_type = 'Person'
                elif 'team' in natural_language_query.lower() or 'club' in natural_language_query.lower():
                    subject_type = 'Team'
            elif query_dict['object'] == '?':
                unknown_component = 'object'
                if 'when' in natural_language_query.lower() or 'year' in natural_language_query.lower():
                    subject_type = 'Date'
            elif query_dict['predicate'] == '?':
                unknown_component = 'predicate'

            # Use turtle file path instead of JSON
            sparql_results = self.llm_verifier.process_query(
                user_query=query_dict['predicate'] if query_dict['predicate'] != '?' else natural_language_query,
                start_node=start_node,
                subgraph_turtle_path=turtle_filepath,
                unknown_component=unknown_component,
                subject_type=subject_type,
                query_structure = modified_query
            )

            pipeline_results['sparql_generation'] = sparql_results

            #print(f" Generated SPARQL queries: {sparql_results['sparql_queries']}")

            # STAGE 5: Handle Subgraph Expansion (if it occurred)
            if sparql_results['subgraph_expanded']:
                print("\n STAGE 5: Subgraph Expansion Detected")
                print("-" * 40)
                print(f"Subgraph expanded with {len(sparql_results['subpart_entities'])} subpart entities")

            pipeline_results['json_files'] = {
                'original_file': json_filepath,
                'subpart_entities_found': sparql_results['subpart_entities']
            }

            if sparql_results.get('expanded_subgraph_path'):
                print(f" Expanded subgraph saved to: {sparql_results['expanded_subgraph_path']}")

            pipeline_results['turtle_files'] = {
                'original_file': turtle_filepath,
                'expanded_file': sparql_results.get('expanded_subgraph_path'),
                'subpart_entities_found': sparql_results['subpart_entities']
            }

            # STAGE 6: Execute SPARQL and get a natural-language answer

            print("\n STAGE 6: Execute SPARQL & Generate Answer ")
            print("-" * 40)

            answers = []
            for i, qblk in enumerate(sparql_results.get('sparql_queries', []), 1):
                qtxt = qblk.get('query')
                if not qtxt:
                    continue

                ans = answer_question_with_sparql(
                      self.rdf_graph,
                      natural_language_query,
                      qtxt,
                      known_entity=start_node,  # Pass the start node
                      unknown_component=unknown_component  # Pass the unknown component
                  )
                answers.append(ans)
                # print(f"🧪 SPARQL #{i} returned {len(ans['rows'])} row(s)")
                # print(f"💬 Answer #{i}: {ans['answer']}")

            pipeline_results['answers'] = answers

            # Set final answer for DataFrame
            if answers:
                pipeline_results['final_answer'] = answers[0]['answer']
            else:
                pipeline_results['final_answer'] = "No answer found"

            print()
            print("PIPELINE COMPLETED SUCCESSFULLY!")

        except Exception as e:
            error_msg = str(e)
            print(f"❌ Pipeline failed at stage: {error_msg}")
            pipeline_results['error'] = error_msg
            pipeline_results['final_answer'] = f"Error: {error_msg}"

        return pipeline_results

    def print_pipeline_summary(self, results: Dict[str, Any]):
        """
        Print a comprehensive summary of pipeline results
        """
        print("="*80)
        print("PIPELINE EXECUTION SUMMARY")
        print("="*80)

        print(f"Original Query: {results['input_query']}")
        print(f"Query Structure: {results.get('query_understanding', {})}")
        print(f"Start Node: {results.get('entity_matching', {}).get('start_node', 'N/A')}")
        print(f"Subgraph Triples: {results.get('subgraph_creation', {}).get('num_triples', 0)}")
        print(f"Subpart Entities: {len(results.get('sparql_generation', {}).get('subpart_entities', []))}")
        print(f"SPARQL Queries Generated: {len(results.get('sparql_generation', {}).get('sparql_queries', []))}")

        turtle_files = results.get('turtle_files', {})
       # print(f"📁 Original Turtle File: {turtle_files.get('original_file', 'N/A')}")
#         if turtle_files.get('expanded_file'):

# #            print(f"📁 Expanded Turtle File: {turtle_files.get('expanded_file')}")

        # Show final natural-language answer (from Stage 6)
        answers = results.get('answers', [])
        if answers:
            print(f"\nFinal Answer: {answers[0]['answer']}")
            print(f"Rows returned: {len(answers[0]['rows'])}")

        # Show generated SPARQL queries
        sparql_queries = results.get('sparql_generation', {}).get('sparql_queries', [])
        for i, query_info in enumerate(sparql_queries, 1):
            print(f"\n SPARQL Query {i}:")
            print(query_info.get('query', 'No query generated'))

        # # Show LLM prompt if k-NN search was used : code for developer reference
        # llm_prompt = results.get('sparql_generation', {}).get('llm_prompt')
        # if llm_prompt:
        #     print(f"\n LLM Prompt Generated in case KNN:")
        #     print(llm_prompt[:500] + "..." if len(llm_prompt) > 500 else llm_prompt)

    def analyze_turtle_subgraph(self, turtle_filepath: str) -> Dict[str, Any]:
        """
        Analyze a turtle subgraph file, for developer verification

        Args:
            turtle_filepath: Path to turtle subgraph file

        Returns:
            Analysis results
        """
        try:
            subgraph = self.load_subgraph_turtle(turtle_filepath)

            subjects = set()
            predicates = set()
            objects = set()

            for s, p, o in subgraph:
                subjects.add(str(s))
                predicates.add(str(p))
                objects.add(str(o))

            analysis = {
                'total_triples': len(subgraph),
                'unique_subjects': len(subjects),
                'unique_predicates': len(predicates),
                'unique_objects': len(objects),
                'subjects_sample': list(subjects)[:5],
                'predicates_sample': list(predicates)[:5],
                'objects_sample': list(objects)[:5]
            }

            # print(f"   Subgraph Analysis for {turtle_filepath}:")
            # print(f"   Total Triples: {analysis['total_triples']}")
            # print(f"   Unique Subjects: {analysis['unique_subjects']}")
            # print(f"   Unique Predicates: {analysis['unique_predicates']}")
            # print(f"   Unique Objects: {analysis['unique_objects']}")

            return analysis

        except Exception as e:
            print(f"Error analyzing turtle subgraph: {e}")
            return {}


def test_pipeline_with_queries():
    """
    Test the pipeline with multiple example queries
    """
    # Initialize pipeline
    rdf_graph_path = "/content/drive/MyDrive/JSON_PIPELINE/output_graph.ttl"
    pipeline = EndToEndPipelineTester(rdf_graph_path)

    # Test queries
    test_queries = [
        "What did Alex Ferguson manage?",
        "Who won the President's Cup in 2007?",
        "What position did Oh My God peak at?",
        "What did Baby Baby become?",
        "Who is the song Oh My God by?"
    ]

    all_results = {}

    # ADDED: DataFrame to collect results
    qa_data = []

    for query in test_queries:
        print(f"\n{'='*100}")
        print(f" TESTING QUERY: {query}")
        print(f"{'='*100}")

        try:
            results = pipeline.run_complete_pipeline(query, max_depth=3)
            all_results[query] = results

            # ADDED: Collect data for DataFrame
            qa_row = {
                'question': query,
                'answer': results.get('final_answer', 'No answer'),
                'error': results.get('error', None)
            }
            qa_data.append(qa_row)

            # Print summary
            pipeline.print_pipeline_summary(results)

            # Analyze turtle files if they exist
            turtle_files = results.get('turtle_files', {})
            if turtle_files.get('original_file'):
                pipeline.analyze_turtle_subgraph(turtle_files['original_file'])
            if turtle_files.get('expanded_file'):
                pipeline.analyze_turtle_subgraph(turtle_files['expanded_file'])

        except Exception as e:
            error_msg = str(e)
            print(f"❌ Test failed for query '{query}': {error_msg}")
            all_results[query] = {'error': error_msg}

            # ADDED: Collect error data for DataFrame
            qa_row = {
                'question': query,
                'answer': f"Pipeline Error: {error_msg}",
                'error': error_msg
            }
            qa_data.append(qa_row)

    # ADDED: Create and save DataFrame
    qa_df = pd.DataFrame(qa_data)

    # Save DataFrame to CSV
    output_path = "/content/drive/MyDrive/JSON_PIPELINE/test_results.csv"
    qa_df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"\n DataFrame saved to: {output_path}")

    # Display DataFrame summary
    print("\n RESULTS SUMMARY:")
    print(qa_df.to_string(index=False))

    return all_results, qa_df



### RUNNING THE TEST QUERY

In [21]:
start_time = time.time()
if __name__ == "__main__":
    # Initialize pipeline
    rdf_graph_path = "/content/drive/MyDrive/JSON_PIPELINE/output_graph.ttl"
    pipeline = EndToEndPipelineTester(rdf_graph_path)

    # ---------- batch run over curated question set ----------
    curated_path = "/content/drive/MyDrive/JSON_PIPELINE/hotpot_small_dataset_extracted/hotpot_small_dataset/questions_set_easy.json"
    with open(curated_path, "r", encoding="utf-8") as fh:
        question_items = json.load(fh)

    batch_results = []

    # ADDED: DataFrame to collect batch results
    batch_qa_data = []

    for item in question_items:
        q = item["Question"]
        print("\n" + "-"*70)
        print("▶", q)

        try:
            res = pipeline.run_complete_pipeline(q)
            batch_results.append(res)

            # ADDED: Collect data for DataFrame
            qa_row = {
                'question': q,
                'answer': res.get('final_answer', 'No answer'),
                'error': res.get('error', None)
            }
            batch_qa_data.append(qa_row)

            pipeline.print_pipeline_summary(res)

        except Exception as e:
            error_msg = str(e)
            print(f"Batch processing failed for question '{q}': {error_msg}")

            # ADDED: Collect error data for DataFrame
            qa_row = {
                'question': q,
                'answer': f"Processing Error: {error_msg}",
                'error': error_msg
            }
            batch_qa_data.append(qa_row)

    # Create and save batch DataFrame - for developer reference
    batch_qa_df = pd.DataFrame(batch_qa_data)
    end_time = time.time()
    elapsed = end_time - start_time
    # Save batch DataFrame to CSV
    batch_output_path = "/content/drive/MyDrive/JSON_PIPELINE/batch_results.csv"
    batch_qa_df.to_csv(batch_output_path, index=False, encoding='utf-8')
    print(f"\n Batch DataFrame saved to: {batch_output_path}")

    # # Display batch DataFrame summary
    # print("\nBATCH RESULTS SUMMARY:")
    # print(batch_qa_df.to_string(index=False))

    # Display success/failure statistics
    total_questions = len(batch_qa_df)
    successful_answers = len(batch_qa_df[batch_qa_df['error'].isna()])
    error_count = len(batch_qa_df[batch_qa_df['error'].notna()])

    print(f"\n QUESTIONS BATCH STATISTICS:")
    print(f"Total Questions: {total_questions}")
    print(f"Successful Answers: {successful_answers}")
    print(f"Errors: {error_count}")
    print(f"Success Rate: {successful_answers/total_questions*100:.1f}%")

    # `batch_results` now holds the output dict for every question
    # `batch_qa_df` now holds the DataFrame with question, answer, error columns

Built global predicate index with 1083 predicates
Pipeline initialized with graph containing 1985 triples

----------------------------------------------------------------------
▶ Who directed The Family Man?
STARTING END-TO-END PIPELINE TESTING
Natural Language Query: Who directed The Family Man?

STAGE 1: Query Understanding
----------------------------------------
Parsed query: {'subject': '?', 'predicate': 'directed_by', 'object': 'The_Family_Man'}

STAGE 2: Entity Matching & Start Node Detection
----------------------------------------
Modified query: {'subject': '?', 'predicate': 'directed_by', 'object': 'The_Family_Man'}

STAGE 3: Subgraph Creation
----------------------------------------
🔍 Searching for: The_Family_Man
  Found as subject: True
  Found as predicate: False
  Found as object (URI): True
  Found as object (Literal): False
  Starting as SUBJECT: http://example.org/resource/The_Family_Man
  Starting as OBJECT (URI): http://example.org/resource/The_Family_Man
Final: 9

Modified query: {'subject': '?', 'predicate': 'scored_goal_against', 'object': '1–0 to Sheffield Wednesday'}

STAGE 3: Subgraph Creation
----------------------------------------
🔍 Searching for: 1–0 to Sheffield Wednesday
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): 1–0 to Sheffield Wednesday
Final: 568 triples | 374 nodes | 295 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/1–0_to_Sheffield_Wednesday.json
Total triples saved: 568
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/1–0_to_Sheffield_Wednesday_subgraph.ttl
Total triples in subgraph: 299

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of '1–0 to Sheffield Wednesday' in subgraph
Summary: 0 URI variants, 1 Literal variants
Total unique variants: 1
Generated SPARQL with variants:
SELECT DISTINCT ?answer WHERE {
    VALUES ?entity 

Modified query: {'subject': '?', 'predicate': 'won', 'object': "the President's Cup in 2007"}

STAGE 3: Subgraph Creation
----------------------------------------
🔍 Searching for: the President's Cup in 2007
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): the President's Cup in 2007
Final: 633 triples | 505 nodes | 477 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/the_President's_Cup_in_2007.json
Total triples saved: 633
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/the_President's_Cup_in_2007_subgraph.ttl
Total triples in subgraph: 487

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of 'the President's Cup in 2007' in subgraph
Summary: 0 URI variants, 1 Literal variants
Total unique variants: 1
Generated SPARQL with variants:
SELECT DISTINCT ?answer WHERE {
    VALUES ?entity { "the Pre

Modified query: {'subject': '?', 'predicate': 'hosted', 'object': 'the boxing match between Muhammad Ali and Sonny Liston in 1965'}

STAGE 3: Subgraph Creation
----------------------------------------
🔍 Searching for: the boxing match between Muhammad Ali and Sonny Liston in 1965
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): the boxing match between Muhammad Ali and Sonny Liston in 1965
Final: 40 triples | 24 nodes | 21 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/the_boxing_match_between_Muhammad_Ali_and_Sonny_Liston_in_1965.json
Total triples saved: 40
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/the_boxing_match_between_Muhammad_Ali_and_Sonny_Liston_in_1965_subgraph.ttl
Total triples in subgraph: 21

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of 'the boxing match between Muham

🔍 Searching for: the Coliseé
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): the Coliseé
Final: 3 triples | 2 nodes | 1 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/the_Coliseé.json
Total triples saved: 3
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/the_Coliseé_subgraph.ttl
Total triples in subgraph: 1

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of 'the Coliseé' in subgraph
Summary: 0 URI variants, 1 Literal variants
Total unique variants: 1
Generated SPARQL with variants:
SELECT DISTINCT ?answer WHERE {
    VALUES ?entity { "the Coliseé" }
    ?entity ?answer "Lewiston" .
}

 STAGE 6: Execute SPARQL & Generate Answer 
----------------------------------------
 ERROR !Original SPARQL returned 0 results
   Trying bidirectional query 1: SELECT DISTINCT ?answer WHERE { <http://example.

🔍 Searching for: in the title role in Ed Wood
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): in the title role in Ed Wood
Final: 242 triples | 190 nodes | 186 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/in_the_title_role_in_Ed_Wood.json
Total triples saved: 242
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/in_the_title_role_in_Ed_Wood_subgraph.ttl
Total triples in subgraph: 193

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of 'in the title role in Ed Wood' in subgraph
Summary: 0 URI variants, 1 Literal variants
Total unique variants: 1
Generated SPARQL with variants:
SELECT DISTINCT ?answer WHERE {
    VALUES ?entity { "in the title role in Ed Wood" }
    ?entity ?answer <http://example.org/resource/Plan_9_from_Outer_Space> .
}

 STAGE 6: Execute SPARQL & Generate Answer 
----------

Modified query: {'subject': 'in the title role in Ed Wood', 'predicate': 'wrote', 'object': '?'}

STAGE 3: Subgraph Creation
----------------------------------------
🔍 Searching for: in the title role in Ed Wood
  Found as subject: False
  Found as predicate: False
  Found as object (URI): False
  Found as object (Literal): True
  Starting as OBJECT (Literal): in the title role in Ed Wood
Final: 242 triples | 190 nodes | 186 edges
Triples saved to: /content/drive/MyDrive/JSON_PIPELINE/in_the_title_role_in_Ed_Wood.json
Total triples saved: 242
Subgraph saved to: /content/drive/MyDrive/JSON_PIPELINE/in_the_title_role_in_Ed_Wood_subgraph.ttl
Total triples in subgraph: 193

 STAGE 4: SPARQL Query Generation with LLM Verification 
----------------------------------------
Searching for variants of 'in the title role in Ed Wood' in subgraph
Summary: 0 URI variants, 1 Literal variants
Total unique variants: 1
Generated SPARQL with variants:
SELECT DISTINCT ?answer WHERE {
    VALUES ?entity { 

In [22]:
print(f"TOTAL TIME TAKEN FOR RUNNING THE PIPELINE ON 12 Questions: {elapsed} seconds")

TOTAL TIME TAKEN FOR RUNNING THE PIPELINE ON 12 Questions: 376.1244285106659 seconds


### RESULT : DataFrame containing all the Natural Language Answers, for the User Queries

In [23]:
# Loading the dataframe and printing in a readable format here
batch_qa_df[::]

,question,answer,error
0,Who directed The Family Man?,The Family Man was directed by Brett Ratner.,None
1,Who scored a 97th-minute goal against Sheffiel...,The team that scored a 97th-minute goal agains...,None
2,Who won the President's Cup in 2007?,The President's Cup in 2007 was won by The Lew...,None
3,What venue hosted the Ali vs. Liston fight in ...,The information provided does not specify the ...,None
4,The Colisée is what to Lewiston?,The information provided does not clearly stat...,None
5,What did Ed Wood do related to Plan 9 from Out...,No facts were found regarding Ed Wood's involv...,None
6,What is the relationship of the Maine Nordique...,The information provided does not clearly stat...,None
7,Who broke up with Dolores Fuller?,Wood broke up with Dolores Fuller.,None
8,Which team did Matthew Wicks join?,Matthew Wicks joined the Arsenal team.,None
9,What did Ed Wood write in 1965?,The information provided does not specify what...,None


---